<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="50%"></a>
</p>

# Line Detection

Line detection in Duckietown is the process of extracting line segments from an image, focusing on specific colors that correspond to lane markers (white, yellow and red).

This process combines multiple stages of color filtering, edge detection and line segment extraction, and is designed to efficiently detect lane lines in Duckietowns.

## Images Preprocessing

The process starts from pre-processing images to facilitate and speed up the process of detecting and extracting colored line segments. Before images are even generated, the camera drivers are configured to produce relatively low resolution images (less pixels to process), with low exposure windows to minimize motion blur.

Once an image reaches the line detector, the following preprocessing steps are furthermore applied:

1. A color correction algorithm is applied to make the fundamental colors: white, yellow and red, more distinguishable. This step includes converting the image to HSV (Hue, Saturation and Value) color space. HSV is preferred over RGB (Red, Green and Blue) for color segmentation, as it separates chromatic content (hue) from intensity (value), making it easier to isolate specific colors
2. The image is downsized to (by default) 120x160 pixels
3. Everything above the horizon is cut off (there are no lanes in the sky)

Then, a color range filter is applied to segment regions corresponding to marker colors (e.g., white, yellow and red). This operation results in a binary mask, highlighting only the regions of interest.

You can view how this filtering process is working using the image viewer:

```bash
  dts duckiebot image_viewer ROBOT_NAME
```

You can look at the colors that are getting detected under the `line_detector_node/debug/maps/jpeg` topic. If things are working well you should see:

![segment maps](../assets/images/maps.png)

## Edge detection

To identify edges, we use a [Canny Edge detection algorithm](https://en.wikipedia.org/wiki/Canny_edge_detector), which fundamentally leverages intensity gradients and a few other tricks (non-maximum suppression and hysterisis thresholding) to identify the boundaries of the lanes. A dedicated [Image Processing Duckietown lecture](https://docs.duckietown.com/ente/duckietown-manual/80-instructor-manual/available-resources/slides/vision/06-full-lecture-vision.html#image-processing) is available for details, but in summary: 

* **Gradient calculation**: The image gradient at every pixel $G = \sqrt{G_x^2 + G_y^2}$ is computed using the [Sobel operator](https://en.wikipedia.org/wiki/Sobel_operator), a finite differences approximation of the image intesity gradient, ideally: $G_x = \frac{\partial I}{\partial x}$ and $G_y = \frac{\partial I}{\partial y}$.

* **Non-maximum suppression**: Edges are thinned out by suppressing the pixels that are not part of the edge contours.

* **Hysteresis thresholding**: Two thresholds ($T_{low}$ and $T_{high}$) are applied to classify edges as strong, weak or irrelevant. Edges stronger than $T_{high}$ are retained and weak edges connected to strong ones are kept.

You can look at the edges being detected under the `line_detector_node/debug/edges/jpeg` topic in the image viewer. If things are working well you should see:

![edges](../assets/images/edges.png)

## Line segment extraction

The detected edges are analyzed to extract line segments using the [Hough line transform](https://en.wikipedia.org/wiki/Hough_transform). This method identifies lines in the image plane by implementing a voting scheme in the $(\rho,\theta)$ space, where:

* $(\rho,\theta)$, definied as $\rho = x\cos(\theta) + y\sin(\theta)$ are a polar parametrization of lines ($y = mx + q$) in the Cartesian plane, that leverages the distance and orientation of the line's normal through the origin thus avoiding singularities for vertical lines. Each previously detected edge "point" $(x, y)$ contributes a vote for all possible lines passing through it.

* These votes are accounted for in an accumulator space, and a defined threshold ensures that only lines with sufficient votes are retained.

You can look at the detected segments under the `line_detector_node/debug/segments/jpeg` topic in the image viewer. If things are working well you should see:

![segment maps](../assets/images/segments.png)


## Tuning the HSV thresholds

To view the colormaps:

1. Run `dts duckiebot image_viewer DUCKIEBOT_NAME`.
2. Select the `NODE/line_detector_node/debug/maps/jpeg` topic, where `NODE` is `DUCKIEBOT_NAME/node/image_relayer`.

If the white or yellow regions of the image are not being well segmented, the color thresholds defining these regions can be tuned using a new `dt-core/packages/line_detector/config/line_detector_node/DUCKIEBOT_NAME.yaml` configuration file.
The default color thresholds are: 

```yaml
colors:
  RED:
    low_1: [0,140,100]
    high_1: [15,255,255]
    low_2: [165,140,100]
    high_2: [180,255,255]
  WHITE:
    low: [0,0,150]
    high: [180,100,255]
  YELLOW:
    low: [15,80,50]
    high: [45,255,255]
```

which are in HSV space, as described above.

**NOTE**:
At this stage, focus only on the `WHITE` and `YELLOW` colors. The may need to be tuned depending on the type and amount of light in your environment. `RED` is inconsequential for this specific lane following autonomous behavior.  

## Tuning the edge and segment extration parameters

In the same config file as above you will also find the following parameters for Canny edge detection and Hough transform: 

```yaml
line_detector_parameters:
  canny_thresholds: [80,200]
  canny_aperture_size: 3
  dilation_kernel_size: 3
  hough_threshold: 2
  hough_min_line_length: 3
  hough_max_line_gap: 1
```

You can experiment with changing these parameters to see the effect on the edge detection or the resulting segments. Below is a short description of their role and meaning.  

### Canny edge detection

 - `canny_thresholds`: [80, 200] are the lower and upper hysteresis thresholds respectively. Edges with gradient strength above 200 are kept, below 80 are discarded, and those in between are only kept if connected to a strong edge.
 - `canny_aperture_size`: 3 is the size of the Sobel kernel used to compute gradients (3 = 3×3). Larger values detect broader/smoother edges.
 - `dilation_kernel_size`: 3. After the Canny detection step, the edge image is dilated with a 3×3 kernel to thicken edges and close small gaps before the Hough transform is applied.


### Hough line detection

 - `hough_threshold`: 2 — minimum number of votes (edge points) a line must accumulate in Hough space to be returned. Lower = more lines detected, including weak ones.
 - `hough_min_line_length`: 3 — shortest line segment (in pixels) that will be reported. Shorter segments are discarded.
 - `hough_max_line_gap`: 1 — maximum gap (in pixels) between two points on the same line before they're treated as separate segments. Higher values join broken lines together.





For much more detail about how image filtering works, refer to the third notebook of the [computer vision learning experience](https://github.com/duckietown/lx-computer-vision/), and/or the [Duckietown Computer Vision lectures](https://docs.duckietown.com/ente/duckietown-manual/80-instructor-manual/available-resources/slides/vision/00-vision-overview.html).

To learn what to do with these newly identified segments, we can move onto the next notebook that details [ground projection](./02_ground_projection.ipynb).